# 15 - Previsão Brasileirão 2026 (Monte Carlo)

Simulação Monte Carlo para prever a classificação final do Brasileirão 2026.
Com base no desempenho atual (rodadas jogadas) e no histórico dos times,
simulamos milhares de cenários para estimar probabilidades de:
- Título
- Libertadores (G6)
- Rebaixamento (Z4)

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

np.random.seed(42)

df = pd.read_csv('../data/raw/campeonato-brasileiro-full.csv')
df['data'] = pd.to_datetime(df['data'], format='%d/%m/%Y')
df['ano'] = df['data'].dt.year
df = df.dropna(subset=['rodata'])
df['rodata'] = df['rodata'].astype(int)

# Fix temporada 2020 (COVID): rodadas 28-38 jogadas em jan-fev/2021 pertencem a temporada 2020
mask_2020 = (df['data'].dt.year == 2021) & (df['rodata'] >= 28) & (df['data'] < '2021-03-01')
df.loc[mask_2020, 'ano'] = 2020

print(f'Total de jogos: {len(df)}')

Total de jogos: 9225


In [2]:
# Parâmetros
ANO_ATUAL = 2026
N_SIMULACOES = 10000
TOTAL_RODADAS = 38

# Separar dados
df_historico = df[df['ano'] < ANO_ATUAL].copy()
df_atual = df[df['ano'] == ANO_ATUAL].copy()

rodada_atual = int(df_atual['rodata'].max())
times_2026 = sorted(set(df_atual['mandante'].unique()) | set(df_atual['visitante'].unique()))

print(f'Rodada atual: {rodada_atual}')
print(f'Times: {len(times_2026)}')
print(f'Rodadas restantes: {TOTAL_RODADAS - rodada_atual}')

Rodada atual: 6
Times: 20
Rodadas restantes: 32


In [3]:
# === CALCULAR FORÇA DE CADA TIME ===
# Combinamos: desempenho atual em 2026 + histórico recente (2022-2025)

def calcular_forca_historica(df_hist, times, anos_recentes=4):
    """Calcula força média de cada time com base no histórico recente."""
    df_rec = df_hist[df_hist['ano'] >= df_hist['ano'].max() - anos_recentes + 1]
    forca = {}
    
    for time in times:
        # Jogos como mandante
        mand = df_rec[df_rec['mandante'] == time]
        # Jogos como visitante
        visit = df_rec[df_rec['visitante'] == time]
        
        jogos = len(mand) + len(visit)
        if jogos == 0:
            forca[time] = {'ataque_m': 1.3, 'defesa_m': 1.1,
                           'ataque_v': 1.0, 'defesa_v': 1.3,
                           'jogos': 0}
            continue
        
        # Médias de gols
        gols_feitos_m = mand['mandante_Placar'].mean() if len(mand) > 0 else 1.3
        gols_sofridos_m = mand['visitante_Placar'].mean() if len(mand) > 0 else 1.1
        gols_feitos_v = visit['visitante_Placar'].mean() if len(visit) > 0 else 1.0
        gols_sofridos_v = visit['mandante_Placar'].mean() if len(visit) > 0 else 1.3
        
        forca[time] = {
            'ataque_m': gols_feitos_m,
            'defesa_m': gols_sofridos_m,
            'ataque_v': gols_feitos_v,
            'defesa_v': gols_sofridos_v,
            'jogos': jogos
        }
    
    return forca


def calcular_classificacao_atual(df_atual):
    """Calcula pontos, vitórias, saldo e gols atuais."""
    times = set(df_atual['mandante'].unique()) | set(df_atual['visitante'].unique())
    stats = {t: {'pts': 0, 'v': 0, 'sg': 0, 'gp': 0, 'j': 0} for t in times}
    
    for _, jogo in df_atual.iterrows():
        m, vis = jogo['mandante'], jogo['visitante']
        gm, gv = int(jogo['mandante_Placar']), int(jogo['visitante_Placar'])
        
        stats[m]['j'] += 1
        stats[vis]['j'] += 1
        stats[m]['gp'] += gm
        stats[vis]['gp'] += gv
        stats[m]['sg'] += gm - gv
        stats[vis]['sg'] += gv - gm
        
        if gm > gv:
            stats[m]['pts'] += 3
            stats[m]['v'] += 1
        elif gv > gm:
            stats[vis]['pts'] += 3
            stats[vis]['v'] += 1
        else:
            stats[m]['pts'] += 1
            stats[vis]['pts'] += 1
    
    return stats


forca = calcular_forca_historica(df_historico, times_2026)
stats_atuais = calcular_classificacao_atual(df_atual)

# Mostrar classificação atual
ranking_atual = sorted(stats_atuais.items(), 
                       key=lambda x: (x[1]['pts'], x[1]['v'], x[1]['sg'], x[1]['gp']),
                       reverse=True)
print(f'Classificação após {rodada_atual} rodadas:')
for i, (t, s) in enumerate(ranking_atual, 1):
    print(f'  {i:2d}. {t:<20s} {s["pts"]:2d} pts  {s["v"]}V  SG={s["sg"]:+d}')

Classificação após 6 rodadas:
   1. Sao Paulo            18 pts  6V  SG=+9
   2. Fluminense           15 pts  5V  SG=+5
   3. Palmeiras            13 pts  4V  SG=+7
   4. Flamengo             12 pts  4V  SG=+7
   5. Coritiba             12 pts  4V  SG=+4
   6. Corinthians          12 pts  4V  SG=+3
   7. Athletico-PR         12 pts  4V  SG=+1
   8. Bragantino            9 pts  3V  SG=-1
   9. Vitoria               9 pts  3V  SG=-2
  10. Mirassol              9 pts  2V  SG=+3
  11. Bahia                 8 pts  2V  SG=+1
  12. Atletico-MG           7 pts  2V  SG=+0
  13. Gremio                7 pts  2V  SG=-1
  14. Chapecoense           7 pts  2V  SG=-5
  15. Botafogo              6 pts  2V  SG=-1
  16. Internacional         6 pts  2V  SG=-4
  17. Vasco                 5 pts  1V  SG=-2
  18. Santos                4 pts  1V  SG=-4
  19. Cruzeiro              3 pts  1V  SG=-9
  20. Remo                  0 pts  0V  SG=-11


In [4]:
# === GERAR TABELA DE JOGOS RESTANTES ===
# No Brasileirão, todos jogam contra todos (ida e volta)
# Precisamos descobrir quais jogos ainda não aconteceram

jogos_feitos = set()
for _, jogo in df_atual.iterrows():
    jogos_feitos.add((jogo['mandante'], jogo['visitante']))

jogos_restantes = []
for m in times_2026:
    for v in times_2026:
        if m != v and (m, v) not in jogos_feitos:
            jogos_restantes.append((m, v))

print(f'Jogos já realizados: {len(jogos_feitos)}')
print(f'Jogos restantes: {len(jogos_restantes)}')
print(f'Total esperado: {20 * 19} (ida e volta)')

Jogos já realizados: 60
Jogos restantes: 320
Total esperado: 380 (ida e volta)


In [5]:
# === SIMULAÇÃO MONTE CARLO ===

# Média geral da liga (para normalização)
media_gols_liga = df_historico['mandante_Placar'].mean()
media_gols_visitante_liga = df_historico['visitante_Placar'].mean()


def simular_jogo(mandante, visitante, forca):
    """Simula um jogo usando distribuição de Poisson."""
    fm = forca.get(mandante, {})
    fv = forca.get(visitante, {})
    
    # Lambda do mandante: sua força de ataque em casa vs defesa do visitante fora
    # Normalizado pela média da liga
    ataque_m = fm.get('ataque_m', 1.3)
    defesa_v = fv.get('defesa_v', 1.3)
    lambda_m = (ataque_m / media_gols_liga) * (defesa_v / media_gols_visitante_liga) * media_gols_liga
    
    # Lambda do visitante
    ataque_v = fv.get('ataque_v', 1.0)
    defesa_m = fm.get('defesa_m', 1.1)
    lambda_v = (ataque_v / media_gols_visitante_liga) * (defesa_m / media_gols_liga) * media_gols_visitante_liga
    
    # Limitar lambdas para evitar valores extremos
    lambda_m = np.clip(lambda_m, 0.3, 4.0)
    lambda_v = np.clip(lambda_v, 0.2, 3.5)
    
    gols_m = np.random.poisson(lambda_m)
    gols_v = np.random.poisson(lambda_v)
    
    return gols_m, gols_v


def simular_campeonato(stats_atuais, jogos_restantes, forca):
    """Simula os jogos restantes e retorna classificação final."""
    # Copiar stats atuais
    stats = {t: dict(s) for t, s in stats_atuais.items()}
    
    for mandante, visitante in jogos_restantes:
        gm, gv = simular_jogo(mandante, visitante, forca)
        
        stats[mandante]['gp'] += gm
        stats[visitante]['gp'] += gv
        stats[mandante]['sg'] += gm - gv
        stats[visitante]['sg'] += gv - gm
        
        if gm > gv:
            stats[mandante]['pts'] += 3
            stats[mandante]['v'] += 1
        elif gv > gm:
            stats[visitante]['pts'] += 3
            stats[visitante]['v'] += 1
        else:
            stats[mandante]['pts'] += 1
            stats[visitante]['pts'] += 1
    
    # Ranking: pontos > vitórias > saldo > gols pró
    ranking = sorted(stats.items(),
                     key=lambda x: (x[1]['pts'], x[1]['v'], x[1]['sg'], x[1]['gp']),
                     reverse=True)
    
    return {time: pos+1 for pos, (time, _) in enumerate(ranking)}, \
           {time: s['pts'] for time, s in stats.items()}


# Rodar simulações
print(f'Rodando {N_SIMULACOES:,} simulações...')
posicoes = {t: [] for t in times_2026}
pontos_finais = {t: [] for t in times_2026}

for i in range(N_SIMULACOES):
    ranking, pts = simular_campeonato(stats_atuais, jogos_restantes, forca)
    for time in times_2026:
        posicoes[time].append(ranking[time])
        pontos_finais[time].append(pts[time])

print('Simulação concluída!')

Rodando 10,000 simulações...


Simulação concluída!


In [6]:
# === CALCULAR PROBABILIDADES ===

resultados = []
for time in times_2026:
    pos_array = np.array(posicoes[time])
    pts_array = np.array(pontos_finais[time])
    
    resultados.append({
        'time': time,
        'pts_atual': stats_atuais[time]['pts'],
        'pos_media': np.mean(pos_array),
        'pos_mediana': np.median(pos_array),
        'pts_media': np.mean(pts_array),
        'pts_min': np.percentile(pts_array, 5),
        'pts_max': np.percentile(pts_array, 95),
        'prob_titulo': (pos_array == 1).mean() * 100,
        'prob_g4': (pos_array <= 4).mean() * 100,
        'prob_g6': (pos_array <= 6).mean() * 100,
        'prob_z4': (pos_array >= 17).mean() * 100,
        'prob_rebaixamento': (pos_array >= 17).mean() * 100,
    })

df_prob = pd.DataFrame(resultados).sort_values('pos_media')

print('Probabilidades (%):')
print(f'{"Time":<20s} {"Pts":>4s} {"PtsPrev":>7s} {"Título":>7s} {"G6":>6s} {"Z4":>6s}')
print('-' * 55)
for _, r in df_prob.iterrows():
    print(f'{r["time"]:<20s} {r["pts_atual"]:4.0f} {r["pts_media"]:7.1f} '
          f'{r["prob_titulo"]:6.1f}% {r["prob_g6"]:5.1f}% {r["prob_z4"]:5.1f}%')

Probabilidades (%):
Time                  Pts PtsPrev  Título     G6     Z4
-------------------------------------------------------
Palmeiras              13    70.6   50.5%  97.4%   0.0%
Flamengo               12    66.6   22.6%  90.8%   0.0%
Sao Paulo              18    63.1    8.0%  77.0%   0.1%
Fluminense             15    62.0    5.8%  71.8%   0.1%
Mirassol                9    62.1    5.7%  71.4%   0.2%
Botafogo                6    61.1    5.5%  65.6%   0.5%
Athletico-PR           12    55.9    0.7%  30.4%   3.1%
Corinthians            12    55.3    0.5%  26.9%   3.9%
Internacional           6    52.7    0.1%  14.7%   7.8%
Atletico-MG             7    52.3    0.3%  13.8%  10.9%
Bahia                   8    51.0    0.0%   7.7%  13.4%
Bragantino              9    50.6    0.1%   8.0%  15.0%
Cruzeiro                3    50.6    0.1%   9.1%  17.1%
Gremio                  7    49.9    0.1%   6.7%  18.6%
Chapecoense             7    47.8    0.0%   4.1%  32.2%
Vitoria                 9   

In [7]:
# === GRÁFICO 1: Probabilidades de Título, Libertadores e Rebaixamento ===

df_plot = df_prob.sort_values('pos_media')

fig = make_subplots(rows=1, cols=3,
                    subplot_titles=['Probabilidade de Título (%)',
                                   'Probabilidade G6 - Libertadores (%)',
                                   'Probabilidade Z4 - Rebaixamento (%)'],
                    horizontal_spacing=0.08)

# Título
titulo_df = df_plot[df_plot['prob_titulo'] > 0.5].sort_values('prob_titulo')
fig.add_trace(go.Bar(
    y=titulo_df['time'], x=titulo_df['prob_titulo'],
    orientation='h', marker_color='#ffd700',
    text=[f'{v:.1f}%' for v in titulo_df['prob_titulo']],
    textposition='outside',
    hovertemplate='%{y}: %{x:.1f}%<extra>Título</extra>'
), row=1, col=1)

# G6
g6_df = df_plot[df_plot['prob_g6'] > 10].sort_values('prob_g6')
fig.add_trace(go.Bar(
    y=g6_df['time'], x=g6_df['prob_g6'],
    orientation='h', marker_color='#238636',
    text=[f'{v:.1f}%' for v in g6_df['prob_g6']],
    textposition='outside',
    hovertemplate='%{y}: %{x:.1f}%<extra>Libertadores</extra>'
), row=1, col=2)

# Z4
z4_df = df_plot[df_plot['prob_z4'] > 5].sort_values('prob_z4')
fig.add_trace(go.Bar(
    y=z4_df['time'], x=z4_df['prob_z4'],
    orientation='h', marker_color='#da3633',
    text=[f'{v:.1f}%' for v in z4_df['prob_z4']],
    textposition='outside',
    hovertemplate='%{y}: %{x:.1f}%<extra>Rebaixamento</extra>'
), row=1, col=3)

fig.update_layout(
    title=f'Previsão Brasileirão 2026 — Após Rodada {rodada_atual}<br>'
          f'<sup>Simulação Monte Carlo ({N_SIMULACOES:,} cenários)</sup>',
    showlegend=False,
    template='plotly_white',
    width=1200, height=500,
)

fig.show()
path = '../charts/previsao_2026.html'
fig.write_html(path, include_plotlyjs='cdn')

# Adiciona descrição estatística
_desc = "Classificação prevista para o Brasileirão 2026 via simulação Monte Carlo (10.000 cenários), com intervalos de confiança para posição final."
with open(path) as f:
    html = f.read()
html = html.replace('<div>', f'<p style="text-align:center;color:#666;font-size:13px;font-family:sans-serif;margin:8px 0 0 0;">{_desc}</p>\n<div>', 1)
with open(path, 'w') as f:
    f.write(html)

print('Gráfico exportado: charts/previsao_2026.html')

Gráfico exportado: charts/previsao_2026.html


In [8]:
# === GRÁFICO 2: Heatmap de distribuição de posições ===

# Matriz: time x posição → probabilidade
ordem_times = df_prob.sort_values('pos_media')['time'].tolist()
matriz = np.zeros((len(ordem_times), 20))

for i, time in enumerate(ordem_times):
    pos_array = np.array(posicoes[time])
    for pos in range(1, 21):
        matriz[i, pos-1] = (pos_array == pos).mean() * 100

fig2 = go.Figure(data=go.Heatmap(
    z=matriz,
    x=[f'{p}°' for p in range(1, 21)],
    y=ordem_times,
    colorscale=[
        [0, '#0d1117'],
        [0.15, '#1a3a1a'],
        [0.4, '#238636'],
        [0.7, '#ffd700'],
        [1.0, '#ff4444']
    ],
    hovertemplate='%{y}<br>Posição: %{x}<br>Probabilidade: %{z:.1f}%<extra></extra>',
    colorbar=dict(title='Prob (%)')
))

# Linhas verticais para G6 e Z4
fig2.add_vline(x=5.5, line_dash='dash', line_color='green', opacity=0.5,
               annotation_text='G6', annotation_position='top')
fig2.add_vline(x=15.5, line_dash='dash', line_color='red', opacity=0.5,
               annotation_text='Z4', annotation_position='top')

fig2.update_layout(
    title=f'Distribuição de Posições Previstas — Brasileirão 2026<br>'
          f'<sup>Após rodada {rodada_atual} | {N_SIMULACOES:,} simulações</sup>',
    xaxis_title='Posição Final',
    yaxis=dict(autorange='reversed'),
    template='plotly_white',
    width=1000, height=700,
)

fig2.show()
path = '../charts/previsao_heatmap.html'
fig2.write_html(path, include_plotlyjs='cdn')

# Adiciona descrição estatística
_desc = "Heatmap de probabilidade de cada time terminar em cada posição, baseado na distribuição empírica das simulações Monte Carlo."
with open(path) as f:
    html = f.read()
html = html.replace('<div>', f'<p style="text-align:center;color:#666;font-size:13px;font-family:sans-serif;margin:8px 0 0 0;">{_desc}</p>\n<div>', 1)
with open(path, 'w') as f:
    f.write(html)

print('Gráfico exportado: charts/previsao_heatmap.html')

Gráfico exportado: charts/previsao_heatmap.html


In [9]:
# === GRÁFICO 3: Faixa de pontuação prevista por time ===

df_pts = df_prob.sort_values('pos_media')

fig3 = go.Figure()

for _, row in df_pts.iterrows():
    time = row['time']
    pts_arr = np.array(pontos_finais[time])
    p5, p25, p50, p75, p95 = np.percentile(pts_arr, [5, 25, 50, 75, 95])
    
    # Barra de intervalo 5-95%
    fig3.add_trace(go.Scatter(
        x=[p5, p95], y=[time, time],
        mode='lines', line=dict(color='#8b949e', width=2),
        showlegend=False,
        hoverinfo='skip'
    ))
    
    # Barra de intervalo 25-75%
    fig3.add_trace(go.Scatter(
        x=[p25, p75], y=[time, time],
        mode='lines', line=dict(color='#58a6ff', width=6),
        showlegend=False,
        hoverinfo='skip'
    ))
    
    # Mediana
    fig3.add_trace(go.Scatter(
        x=[p50], y=[time],
        mode='markers', marker=dict(color='#ffd700', size=10, symbol='diamond'),
        showlegend=False,
        hovertemplate=f'{time}<br>Mediana: {p50:.0f} pts<br>'
                      f'Faixa 50%: {p25:.0f}-{p75:.0f}<br>'
                      f'Faixa 90%: {p5:.0f}-{p95:.0f}<extra></extra>'
    ))

# Linhas de referência históricas
fig3.add_vline(x=45, line_dash='dot', line_color='red', opacity=0.4,
               annotation_text='~Rebaixamento', annotation_position='top left')
fig3.add_vline(x=60, line_dash='dot', line_color='green', opacity=0.4,
               annotation_text='~Libertadores', annotation_position='top left')

fig3.update_layout(
    title=f'Faixa de Pontuação Prevista — Brasileirão 2026<br>'
          f'<sup>Losango = mediana | Barra azul = 50% dos cenários | '
          f'Barra cinza = 90% dos cenários</sup>',
    xaxis_title='Pontos',
    yaxis=dict(autorange='reversed'),
    template='plotly_white',
    width=900, height=700,
)

fig3.show()
path = '../charts/previsao_pontos.html'
fig3.write_html(path, include_plotlyjs='cdn')

# Adiciona descrição estatística
_desc = "Box plot da distribuição de pontuação prevista — losango indica a mediana, barra azul o intervalo interquartil (50%) e barra cinza o intervalo de 90%."
with open(path) as f:
    html = f.read()
html = html.replace('<div>', f'<p style="text-align:center;color:#666;font-size:13px;font-family:sans-serif;margin:8px 0 0 0;">{_desc}</p>\n<div>', 1)
with open(path, 'w') as f:
    f.write(html)

print('Gráfico exportado: charts/previsao_pontos.html')

Gráfico exportado: charts/previsao_pontos.html


In [10]:
# === RESUMO FINAL ===

print(f'\n{"=" * 60}')
print(f'PREVISÃO BRASILEIRÃO 2026 — Após Rodada {rodada_atual}')
print(f'{"=" * 60}')

print(f'\nFavoritos ao TÍTULO:')
for _, r in df_prob.nlargest(5, 'prob_titulo').iterrows():
    if r['prob_titulo'] > 0.5:
        print(f'  {r["time"]:<20s} {r["prob_titulo"]:5.1f}%')

print(f'\nMais provável LIBERTADORES (G6):')
for _, r in df_prob.nlargest(8, 'prob_g6').iterrows():
    if r['prob_g6'] > 20:
        print(f'  {r["time"]:<20s} {r["prob_g6"]:5.1f}%')

print(f'\nMaior risco de REBAIXAMENTO (Z4):')
for _, r in df_prob.nlargest(6, 'prob_z4').iterrows():
    if r['prob_z4'] > 5:
        print(f'  {r["time"]:<20s} {r["prob_z4"]:5.1f}%')

print(f'\n{N_SIMULACOES:,} simulações realizadas.')


PREVISÃO BRASILEIRÃO 2026 — Após Rodada 6

Favoritos ao TÍTULO:
  Palmeiras             50.5%
  Flamengo              22.6%
  Sao Paulo              8.0%
  Fluminense             5.8%
  Mirassol               5.7%

Mais provável LIBERTADORES (G6):
  Palmeiras             97.4%
  Flamengo              90.8%
  Sao Paulo             77.0%
  Fluminense            71.8%
  Mirassol              71.4%
  Botafogo              65.6%
  Athletico-PR          30.4%
  Corinthians           26.9%

Maior risco de REBAIXAMENTO (Z4):
  Remo                  71.7%
  Santos                64.3%
  Coritiba              53.9%
  Vasco                 50.8%
  Vitoria               36.2%
  Chapecoense           32.2%

10,000 simulações realizadas.
